# Configs — 02: Bulk QuickStart Config Apply

**Persona:** Admin

**Goal:** Browse the platform's QuickStart config templates, apply a complete
`routing + driver + feature_type` configuration atomically via the bulk endpoint,
then verify all keys are present and the config is searchable.

---

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- Target catalog and collection already exist (physical storage ready)
- `DYNASTORE_ADMIN_TOKEN` is set

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_ADMIN_TOKEN` | _(empty)_ | Bearer token for admin operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `sentinel2-l2a` | Target collection |

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN   = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30)

print(f"Base URL      : {BASE_URL}")
print(f"Catalog       : {CATALOG_ID}")
print(f"Collection    : {COLLECTION_ID}")
print(f"Auth header   : {'set' if ADMIN_TOKEN else 'not set'}")

In [ ]:
# Step 1 — List all available QuickStart templates
resp = client.get("/configs/examples")
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

examples_payload = resp.json()
templates = (
    examples_payload
    if isinstance(examples_payload, list)
    else examples_payload.get("examples", examples_payload.get("templates", []))
)

print(f"QuickStart templates ({len(templates)}):")
for t in templates:
    name = t.get("name", t.get("id", t)) if isinstance(t, dict) else t
    desc = t.get("description", "") if isinstance(t, dict) else ""
    print(f"  {name:<40}  {desc}")

In [ ]:
# Step 2 — Inspect the routing template
resp = client.get("/configs/examples/routing")
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

routing_template = resp.json()
print("routing template structure:")
print(json.dumps(routing_template, indent=2))

In [ ]:
# Step 3 — Bulk apply: routing + driver:postgresql + feature_type atomically
#
# The bulk endpoint applies all configs in a single transaction.
# If any config fails schema validation, the entire apply is rolled back
# and the errors array in the response lists which keys failed.
quickstart_config_set = {
    "configs": {
        "driver:postgresql": {
            "enabled": True,
            "partition_type": "LIST",
            "sidecars": ["item_metadata", "stac_metadata"],
        },
        "routing": {
            "enabled": True,
            "operations": {
                "WRITE": [{"driver": "driver:postgresql", "on_failure": "fatal"}],
                "READ":  [{"driver": "driver:postgresql", "on_failure": "fatal"}],
            },
        },
        "feature_type": {
            "enabled": True,
            "geometry_type": "Polygon",
        },
    }
}

resp = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/bulk",
    json=quickstart_config_set,
)
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

bulk_response = resp.json()
applied = bulk_response.get("applied", [])
errors  = bulk_response.get("errors", [])

print(f"Bulk apply result:")
print(f"  Applied : {applied}")
print(f"  Errors  : {errors}")

assert len(errors) == 0, f"Bulk apply had errors: {errors}"
print()
print(json.dumps(bulk_response, indent=2))

In [ ]:
# Step 4 — Verify all applied config keys are present
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs"
)
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

all_configs = resp.json()
config_keys = (
    [c.get("plugin_id", c.get("id", c)) for c in all_configs]
    if isinstance(all_configs, list)
    else list(all_configs.get("configs", {}).keys())
)

expected_keys = {"driver:postgresql", "routing", "feature_type"}
missing = expected_keys - set(config_keys)
assert not missing, f"Keys missing after bulk apply: {missing}"

print(f"All expected config keys present: {sorted(expected_keys)}")
print(f"Full config key list: {sorted(config_keys)}")

In [ ]:
# Step 5 — Search configs by query string
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/search",
    params={"q": "routing"},
)
assert resp.status_code == 200, (
    f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
)

search_result = resp.json()
matches = (
    search_result
    if isinstance(search_result, list)
    else search_result.get("results", search_result.get("configs", []))
)

print(f"Search q='routing' → {len(matches)} result(s):")
for m in matches:
    plugin_id = m.get("plugin_id", m.get("id", str(m))) if isinstance(m, dict) else str(m)
    print(f"  {plugin_id}")

## Edge cases

### Case A — One invalid config in a bulk apply

If a single entry in the bulk set fails schema validation, the platform should
populate the `errors` array and (depending on deployment policy) either roll back
all applied configs or report partial success. The cell below demonstrates this
with an invalid `geometry_type` value.

In [ ]:
# One invalid config in a bulk apply — errors array must be non-empty
invalid_bulk = {
    "configs": {
        "driver:postgresql": {
            "enabled": True,
            "partition_type": "LIST",
        },
        "feature_type": {
            "enabled": True,
            # Invalid geometry_type value — should fail schema validation
            "geometry_type": "INVALID_GEOM_TYPE_XYZ",
        },
    }
}

resp = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/bulk",
    json=invalid_bulk,
)

print(f"Bulk apply with invalid feature_type → {resp.status_code}")

if resp.status_code == 200:
    body = resp.json()
    errors = body.get("errors", [])
    assert len(errors) > 0, (
        f"Expected errors array to be non-empty for invalid config, got: {body}"
    )
    print(f"  errors (non-empty as expected): {errors}")
    applied = body.get("applied", [])
    print(f"  applied: {applied}")
    if not applied:
        print("  Rollback behavior: no configs were applied (atomic)")
    else:
        print("  Partial success behavior: valid configs were applied despite one error")
elif resp.status_code == 422:
    print("  422 — platform rejects the entire bulk body on first validation failure (strict mode)")
    detail = resp.json().get("detail", resp.text[:300])
    print(f"  Detail: {detail}")
else:
    print(f"  Unexpected status: {resp.text[:300]}")

# Restore valid state
client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/bulk",
    json=quickstart_config_set,
)

## Teardown

Delete all configs applied by the bulk operation in this notebook.

In [ ]:
for key in ["driver:postgresql", "routing", "feature_type"]:
    resp = client.delete(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/{key}"
    )
    if resp.status_code == 204:
        print(f"Deleted {key!r} — 204")
    elif resp.status_code == 404:
        print(f"  {key!r} not found (already absent)")
    else:
        print(f"  {key!r}: unexpected status {resp.status_code}: {resp.text[:200]}")

client.close()